# XGBoost Optuna Training

Train XGBoost models for `crop_type` and `phenophase_name` using full spectral/index features, cyclic date features, and location features. The pipeline includes class distribution checks, outlier clipping, normalization, Optuna tuning with macro F1, sample weights for imbalance, and saved model artifacts.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys

import joblib
import pandas as pd
from IPython.display import display

NOTEBOOK_ROOT = Path.cwd()
CODE_DIR = NOTEBOOK_ROOT if (NOTEBOOK_ROOT / "features_with_labels_dropna.csv").exists() else NOTEBOOK_ROOT / "Code"
sys.path.insert(0, str(CODE_DIR))

from train_xgboost_optuna import (
    load_dataset,
    prepare_features,
    write_distribution,
    split_indices,
    fit_transform_features,
    encode_target,
    train_and_evaluate,
    set_seed,
)

DATA_PATH = CODE_DIR / "features_with_labels_dropna.csv"
OUTPUT_DIR = CODE_DIR / "xgboost_notebook_full_date_location"
TARGETS = ["crop_type", "phenophase_name"]

TRIALS = 25
TIMEOUT = 600
SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
CLIP_LOWER = 0.01
CLIP_UPPER = 0.99

INCLUDE_DATE_FEATURES = True
INCLUDE_LOCATION = True
INCLUDE_REGION = False
NORMALIZE = True

set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

Data: d:\!Reno\AIG\Code\features_with_labels_dropna.csv
Outputs: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location


## Load Data And Build Features

This uses all `B_*` bands, all configured vegetation/water/built-up indices, `days_to_image`, cyclic date features from `phenophase_date` and `image_date`, plus `Longitude` and `Latitude`.

In [2]:
df = load_dataset(DATA_PATH)
feature_df, feature_cols = prepare_features(
    df=df,
    include_location=INCLUDE_LOCATION,
    include_region=INCLUDE_REGION,
    add_dates=INCLUDE_DATE_FEATURES,
)

clean_mask = df[TARGETS].notna().all(axis=1) & feature_df.notna().all(axis=1)
df = df.loc[clean_mask].reset_index(drop=True)
feature_df = feature_df.loc[clean_mask].reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
print(f"Feature count: {len(feature_cols)}")
display(pd.Series(feature_cols, name="feature").to_frame())
display(df.head())

Rows after cleaning: 4694
Feature count: 29


,feature
0,days_to_image
1,B_B01
2,B_B02
3,B_B03
4,B_B04
5,B_B05
6,B_B06
7,B_B07
8,B_B08
9,B_B09


,point_id,Longitude,Latitude,phenophase_date,image_date,days_to_image,region,crop_type,phenophase_name,B_B01,...,B_B09,B_B11,B_B12,B_B8A,NDVI,NDBI,NDMI,GNDVI,EVI,MNDWI
0,1,125.52644,49.339533,2018/6/7,2018-05-29,9,region09,soybean,Greenup,0.0645,...,0.1924,0.2362,0.1908,0.2192,0.491246,0.041906,-0.041906,0.481583,0.297246,-0.513133
1,1,125.52644,49.339533,2018/6/30,2018-06-28,2,region09,soybean,MidGreenup,31.0000,...,14.0000,49.0000,28.0000,75.0000,0.573333,-0.092593,0.092593,0.475000,-6.515152,-0.400000
2,1,125.52644,49.339533,2018/8/6,2018-07-31,6,region09,soybean,Peak,0.0262,...,0.5686,0.1975,0.0924,0.4653,0.888993,-0.340678,0.340678,0.796064,0.704068,-0.624846
3,1,125.52644,49.339533,2018/7/22,2018-07-26,4,region09,soybean,Maturity,0.0249,...,0.5670,0.2107,0.0926,0.4405,0.895717,-0.318013,0.318013,0.795810,0.701882,-0.639689
4,1,125.52644,49.339533,2018/9/12,2018-09-09,3,region09,soybean,MidSenescence,0.0194,...,0.3005,0.1745,0.0903,0.2956,0.823400,-0.173573,0.173573,0.759943,0.455230,-0.675468


## Distribution And Feature Summary

In [3]:
write_distribution(df, feature_df, OUTPUT_DIR)

for target in TARGETS:
    dist = pd.DataFrame({
        "count": df[target].value_counts(),
        "ratio": df[target].value_counts(normalize=True),
    })
    print(f"\n{target} distribution")
    display(dist)

feature_summary = feature_df.describe().T
display(feature_summary)
print(f"Distribution and feature summary saved under: {OUTPUT_DIR / 'analysis'}")


crop_type distribution


,count,ratio
crop_type,,
rice,2133,0.454410
corn,1371,0.292075
soybean,1190,0.253515



phenophase_name distribution


,count,ratio
phenophase_name,,
Greenup,707,0.150618
MidSenescence,705,0.150192
Dormancy,691,0.147209
Maturity,671,0.142948
MidGreenup,654,0.139327
Senescence,649,0.138262
Peak,617,0.131444


,count,mean,std,min,25%,50%,75%,max
days_to_image,4694.0,4.942267,4.730796,0.000000,2.000000,4.000000,7.000000e+00,42.000000
B_B01,4694.0,0.347369,3.050129,0.000000,0.022500,0.031100,5.080000e-02,34.000000
B_B02,4694.0,0.296668,2.447366,0.000000,0.028300,0.039900,6.170000e-02,29.000000
B_B03,4694.0,0.304192,2.302770,0.009500,0.049300,0.063300,8.697500e-02,28.000000
B_B04,4694.0,0.265317,1.941116,0.004700,0.028725,0.056400,1.016000e-01,28.000000
B_B05,4694.0,0.371610,2.586491,0.011800,0.074725,0.100850,1.460500e-01,36.000000
B_B06,4694.0,0.683743,4.702681,0.011400,0.157400,0.232950,2.949750e-01,74.000000
B_B07,4694.0,0.810004,5.467108,0.011900,0.178225,0.279000,3.921000e-01,88.000000
B_B08,4694.0,0.806183,5.324031,0.017000,0.189850,0.300550,3.980000e-01,86.000000
B_B09,4694.0,0.450417,1.275878,0.011900,0.204950,0.315900,4.183000e-01,19.000000


Distribution and feature summary saved under: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\analysis


## Train/Validation/Test Split And Normalization

Outlier clipping bounds and `StandardScaler` are fit on the training split only, then applied to validation and test splits.

In [4]:
train_idx, val_idx, test_idx = split_indices(
    df=df,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    seed=SEED,
)

x_train, x_val, x_test, scaler, clip_bounds = fit_transform_features(
    feature_df=feature_df,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    clip_lower=CLIP_LOWER,
    clip_upper=CLIP_UPPER,
    normalize=NORMALIZE,
)

preprocessing_dir = OUTPUT_DIR / "preprocessing"
preprocessing_dir.mkdir(parents=True, exist_ok=True)
pd.Series(feature_cols, name="feature").to_csv(preprocessing_dir / "feature_columns.csv", index=False)
clip_bounds.to_csv(preprocessing_dir / "clip_bounds.csv")
if scaler is not None:
    joblib.dump(scaler, preprocessing_dir / "standard_scaler.joblib")

print(f"Split sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
print(f"x_train shape: {x_train.shape}")
print(f"Saved preprocessing artifacts to: {preprocessing_dir}")
display(pd.DataFrame(x_train, columns=feature_cols).describe().T)

Split sizes: train=3004, val=751, test=939
x_train shape: (3004, 29)
Saved preprocessing artifacts to: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\preprocessing


,count,mean,std,min,25%,50%,75%,max
days_to_image,3004.0,-3.492150e-09,1.000166,-1.141763,-0.669201,-0.196639,0.512204,3.820138
B_B01,3004.0,-3.809618e-09,1.000167,-0.437916,-0.305569,-0.238640,-0.087764,7.235031
B_B02,3004.0,-5.079490e-09,1.000166,-0.472908,-0.342006,-0.243177,-0.056500,7.088887
B_B03,3004.0,0.000000e+00,1.000167,-0.601352,-0.375050,-0.245208,-0.025725,6.982049
B_B04,3004.0,1.269873e-09,1.000167,-0.674172,-0.535866,-0.277683,0.134234,6.554642
B_B05,3004.0,-3.809618e-09,1.000167,-0.853333,-0.494034,-0.266148,0.126841,6.604018
B_B06,3004.0,-2.222277e-09,1.000167,-1.457834,-0.689422,-0.108007,0.394080,4.965311
B_B07,3004.0,-6.349363e-10,1.000165,-1.512497,-0.792072,-0.144206,0.600238,3.862572
B_B08,3004.0,2.539745e-09,1.000167,-1.629454,-0.808766,-0.076098,0.598003,3.655754
B_B09,3004.0,-1.904809e-09,1.000167,-1.322980,-0.614487,-0.122791,0.339416,5.554308


## Tune, Train, Evaluate, And Save Models

The Optuna objective is validation macro F1. Each final XGBoost model is refit on train + validation and evaluated on the held-out test set.

In [5]:
args = SimpleNamespace(trials=TRIALS, timeout=TIMEOUT, seed=SEED)
results = []

for target in TARGETS:
    (
        y_train,
        y_val,
        y_test,
        encoder,
        class_weight,
        train_weight,
        val_weight,
    ) = encode_target(
        df=df,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        target=target,
    )

    print("=" * 80)
    print(f"Target: {target}")
    print(f"Classes: {list(encoder.classes_)}")
    print(f"Class weights: {class_weight}")

    result = train_and_evaluate(
        target=target,
        x_train=x_train,
        x_val=x_val,
        x_test=x_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        train_weight=train_weight,
        val_weight=val_weight,
        encoder=encoder,
        class_weight=class_weight,
        feature_cols=feature_cols,
        args=args,
        output_dir=OUTPUT_DIR,
    )
    results.append(result)
    print(f"{target}: macro_f1={result['test_macro_f1']:.4f}, accuracy={result['test_accuracy']:.4f}\n")

results_df = pd.DataFrame(results).sort_values("test_macro_f1", ascending=False)
results_df.to_csv(OUTPUT_DIR / "model_results_summary.csv", index=False)
display(results_df)

[I 2026-05-02 20:57:28,830] A new study created in memory with name: crop_type_xgboost_macro_f1


Target: crop_type
Classes: ['corn', 'rice', 'soybean']
Class weights: {0: 1.1417711896617255, 1: 0.7330405075646657, 2: 1.3158125273762593}


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-05-02 20:57:29,897] Trial 0 finished with value: 0.9774502127215307 and parameters: {'n_estimators': 799, 'learning_rate': 0.10260217747726169, 'max_depth': 8, 'min_child_weight': 4.550475813202184, 'subsample': 0.6202083881990965, 'colsample_bytree': 0.6201975341512912, 'gamma': 3.200866785899844e-08, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598, 'max_bin': 200}. Best is trial 0 with value: 0.9774502127215307.
[I 2026-05-02 20:57:30,289] Trial 1 finished with value: 0.9774498361740358 and parameters: {'n_estimators': 232, 'learning_rate': 0.10905623135414558, 'max_depth': 9, 'min_child_weight': 1.094334266006264, 'subsample': 0.6318212352431953, 'colsample_bytree': 0.6325320294340453, 'gamma': 4.4319427891510175e-06, 'reg_alpha': 0.00052821153945323, 'reg_lambda': 7.71800699380605e-05, 'max_bin': 120}. Best is trial 0 with value: 0.9774502127215307.
[I 2026-05-02 20:57:31,653] Trial 2 finished with value: 0.9712831747157508 and parameters: {'n_estimators

[I 2026-05-02 20:58:05,248] A new study created in memory with name: phenophase_name_xgboost_macro_f1


crop_type: macro_f1=0.9865, accuracy=0.9883

Target: phenophase_name
Classes: ['Dormancy', 'Greenup', 'Maturity', 'MidGreenup', 'MidSenescence', 'Peak', 'Senescence']
Class weights: {0: 0.9731130547457079, 1: 0.9494310998735778, 2: 0.9980066445182725, 3: 1.0266575529733424, 4: 0.9494310998735778, 5: 1.0864376130198914, 6: 1.0315934065934067}


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-05-02 20:58:06,430] Trial 0 finished with value: 0.9930761450687118 and parameters: {'n_estimators': 799, 'learning_rate': 0.10260217747726169, 'max_depth': 8, 'min_child_weight': 4.550475813202184, 'subsample': 0.6202083881990965, 'colsample_bytree': 0.6201975341512912, 'gamma': 3.200866785899844e-08, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598, 'max_bin': 200}. Best is trial 0 with value: 0.9930761450687118.
[I 2026-05-02 20:58:06,866] Trial 1 finished with value: 0.9944696673321659 and parameters: {'n_estimators': 232, 'learning_rate': 0.10905623135414558, 'max_depth': 9, 'min_child_weight': 1.094334266006264, 'subsample': 0.6318212352431953, 'colsample_bytree': 0.6325320294340453, 'gamma': 4.4319427891510175e-06, 'reg_alpha': 0.00052821153945323, 'reg_lambda': 7.71800699380605e-05, 'max_bin': 120}. Best is trial 1 with value: 0.9944696673321659.
[I 2026-05-02 20:58:09,212] Trial 2 finished with value: 0.99722371967655 and parameters: {'n_estimators':

,target,model,test_macro_f1,test_weighted_f1,test_accuracy,best_val_macro_f1,classes
1,phenophase_name,xgboost_optuna,0.993344,0.993605,0.993610,0.997224,"Dormancy, Greenup, Maturity, MidGreenup, MidSe..."
0,crop_type,xgboost_optuna,0.986546,0.988294,0.988285,0.980688,"corn, rice, soybean"


## Saved Artifacts

In [6]:
for target in TARGETS:
    target_dir = OUTPUT_DIR / target
    print(f"{target} model: {target_dir / 'xgboost_model.joblib'}")
    print(f"{target} encoder: {target_dir / 'label_encoder.joblib'}")
    print(f"{target} report: {target_dir / 'classification_report.txt'}")
    print(f"{target} feature importance: {target_dir / 'feature_importance.csv'}")
print(f"Preprocessing: {OUTPUT_DIR / 'preprocessing'}")
print(f"Summary: {OUTPUT_DIR / 'model_results_summary.csv'}")

crop_type model: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\crop_type\xgboost_model.joblib
crop_type encoder: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\crop_type\label_encoder.joblib
crop_type report: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\crop_type\classification_report.txt
crop_type feature importance: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\crop_type\feature_importance.csv
phenophase_name model: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\phenophase_name\xgboost_model.joblib
phenophase_name encoder: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\phenophase_name\label_encoder.joblib
phenophase_name report: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\phenophase_name\classification_report.txt
phenophase_name feature importance: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\phenophase_name\feature_importance.csv
Preprocessing: d:\!Reno\AIG\Code\xgboost_notebook_full_date_location\preprocessing
Summary